# PlantSR · Integrated app: one evaluation for all four tasks
AAI 521 Group 1 · Google Colab · **GPU runtime (T4 or better)**

This notebook checks the integrated system before the report and user testing:

| Step | Section |
|---|---|
| Clone the integration branch and install the app | 1 |
| Install everyone's fine-tuned weights (Jinyuan: denoising, colorization · Peng: super-resolution · Ali: inpainting) | 2–4 |
| Run the validation set through the app's `restore()` and compare with each owner's reported numbers | 5 |
| Optional: the final test-split run (only once the models are frozen) | 6 |
| Visual check, automated tests, and the web app with an API smoke test (each task, each SR model, one pipeline) | 7–9 |
| Save the results to Drive | 10 |

Each task is scored with its owner's conventions: RGB PSNR and SSIM on the 256 × 256 crops (Jinyuan: `data_range=255`; Peng: float images, Gaussian SSIM σ = 1.5), mean CIEDE2000 for colorization, and masked-region metrics for inpainting. If section 5 matches, the app runs the same models as the training notebooks, so the report can quote one set of results.

Everything worth keeping goes to `MyDrive/AAI521/Final_project/app_integration/`. A full run takes about 25 minutes on a T4 (most of it Stable Diffusion and HAT-L).

## 1 · Clone the integration branch and install

In [ ]:
# Clone the integration branch and install the app's libraries (about 3 minutes).
import importlib.util
import os
import subprocess
import sys
from pathlib import Path

env = os.environ.get
REPO_URL = "https://github.com/pwang4-LBNL/AAI521-Group1-PlantSR-Restoration.git"
BRANCH = env("PLANTSR_BRANCH", "feat/app")  # use "main" after the pull request is merged
CODE_DIR = Path(env("PLANTSR_CODE_DIR", "/content/AAI521-Group1-PlantSR-Restoration"))

if env("PLANTSR_SKIP_CLONE") != "1":
    if (CODE_DIR / ".git").is_dir():
        subprocess.run(["git", "-C", str(CODE_DIR), "fetch", "--quiet", "origin", BRANCH], check=True)
        subprocess.run(["git", "-C", str(CODE_DIR), "checkout", "--quiet", "-B", BRANCH, f"origin/{BRANCH}"], check=True)
    else:
        subprocess.run(["git", "clone", "--quiet", "-b", BRANCH, REPO_URL, str(CODE_DIR)], check=True)
COMMIT = subprocess.run(["git", "-C", str(CODE_DIR), "log", "-1", "--format=%h %s"],
                        capture_output=True, text=True).stdout.strip() or "(not a git checkout)"
print("Code:", CODE_DIR, "·", COMMIT)
os.chdir(CODE_DIR)
if str(CODE_DIR) not in sys.path:
    sys.path.insert(0, str(CODE_DIR))

if env("PLANTSR_SKIP_INSTALL") != "1":
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "-r", "app/requirements.txt"], check=True)

from importlib.metadata import PackageNotFoundError, version
from packaging.version import Version

try:  # Colab's old torchao breaks peft (LoRA); the project does not use torchao
    if Version(version("torchao")) < Version("0.16.0"):
        subprocess.run([sys.executable, "-m", "pip", "uninstall", "--quiet", "-y", "torchao"], check=True)
        importlib.invalidate_caches()
except PackageNotFoundError:
    pass

import PIL

if PIL.__version__ != version("Pillow"):
    # Pillow is pinned to 11.3.0 (the SR notebook's version) so the bicubic benchmark inputs match exactly.
    print(f"Pillow {version('Pillow')} was installed but {PIL.__version__} is loaded. "
          "Restarting the runtime; then run the notebook again from the top.")
    os.kill(os.getpid(), 9)

for package in ("torch", "Pillow", "spandrel", "spandrel_extra_arches", "kornia", "diffusers", "peft", "flask"):
    try:
        print(f"{package:22s} {version(package)}")
    except PackageNotFoundError:
        print(f"{package:22s} (not installed)")

## 2 · Mount Drive and configure paths

In [ ]:
# Mount Drive, define paths, and find the teammates' weights in Drive.
import csv
import json
import shutil
import time
import zipfile

try:
    IN_COLAB = importlib.util.find_spec("google.colab") is not None
except ModuleNotFoundError:
    IN_COLAB = False
if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")

PROJECT_DIR = Path(env("PLANTSR_PROJECT_DIR", "/content/drive/MyDrive/AAI521/Final_project"))
OUTPUT_DIR = PROJECT_DIR / "app_integration"  # persisted in Drive
LOCAL_DIR = Path(env("PLANTSR_LOCAL_DIR", "/content/plantsr"))  # fast Colab disk
PREPARED_DIR = LOCAL_DIR / "prepared"
TEAM_ZIP = PROJECT_DIR / "PlantSR_prepared_for_team.zip"
TEAM_ZIP_DRIVE_ID = "1j_vx9Yuow1pu8MzYmbGY4udXBpB4dBx1"  # Peng's shared PlantSR_prepared_for_team.zip
WEEK1_DIR = PROJECT_DIR / "inpainting_week1"
MASKS_ZIP = WEEK1_DIR / "PlantSR_inpainting_masks_v1.zip"
SELECTION_CSV = WEEK1_DIR / "model_selection_val.csv"

SEED = 42
APP_PORT = int(env("PLANTSR_APP_PORT", "7860"))
PARITY_LIMIT = int(env("PLANTSR_PARITY_LIMIT", "0"))  # 0 = every validation crop (needed for the comparison)
N_INPAINT = int(env("PLANTSR_N_INPAINT", "8"))        # validation crops for the Stable Diffusion check
RUN_TEST = env("PLANTSR_RUN_TEST", "0") == "1"        # section 6: set True only when every model is final

for folder in (OUTPUT_DIR, LOCAL_DIR):
    folder.mkdir(parents=True, exist_ok=True)


def find_weights(variable, folder_name, file_name="best.pth"):
    """A path from the environment, else <folder_name>/<file_name> or <folder_name>*.zip near PROJECT_DIR."""
    if env(variable):
        return Path(env(variable))
    for base in (PROJECT_DIR, *sorted(p for p in PROJECT_DIR.iterdir() if p.is_dir())):
        if (base / folder_name / file_name).is_file():
            return base / folder_name / file_name
        zips = sorted(base.glob(f"{folder_name}*.zip"))
        if zips:
            return zips[-1]
    return None


# Point these at the files in your Drive if the search does not find them.
DENOISE_SRC = find_weights("PLANTSR_DENOISE_SRC", "Denoise")      # Jinyuan: Denoise/best.pth
COLORIZE_SRC = find_weights("PLANTSR_COLORIZE_SRC", "Colorize")   # Jinyuan: Colorize/best.pth
SR_SRC = find_weights("PLANTSR_SR_SRC", "PlantSR_x4_app", "plantsr_sr/manifest.json")  # Peng's bundle
if SR_SRC is not None and SR_SRC.name == "manifest.json":
    SR_SRC = SR_SRC.parents[1]
INPAINT_LORA_SRC = env("PLANTSR_INPAINT_LORA_SRC")  # Ali: leave unset until the full LoRA fine-tune (C2) is done

for name, value in (("Denoising", DENOISE_SRC), ("Colorization", COLORIZE_SRC),
                    ("Super-resolution", SR_SRC or "download from Peng's Drive link"),
                    ("Inpainting LoRA", INPAINT_LORA_SRC or "not installed (pretrained SD 1.5 is the default)")):
    print(f"{name:17s} {value}")
print(f"Outputs           {OUTPUT_DIR}")

In [ ]:
# Check the GPU.
import torch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
if DEVICE == "cuda":
    props = torch.cuda.get_device_properties(0)
    GPU = f"{props.name} · {props.total_memory / 1024**3:.1f} GB · compute capability {props.major}.{props.minor}"
else:
    GPU = "none (CPU)"
    print("No GPU detected. Choose Runtime → Change runtime type → T4 GPU, then rerun from the top.")
print("GPU", GPU)

## 3 · Restore the shared dataset and the fixed inpainting masks

In [ ]:
# Extract the team dataset and the Week 1 masks to local disk.
from IPython.display import Markdown, display

if not (PREPARED_DIR / "summary.json").is_file():
    zip_path = TEAM_ZIP
    if not zip_path.is_file():
        import gdown
        zip_path = LOCAL_DIR / TEAM_ZIP.name
        gdown.download(id=TEAM_ZIP_DRIVE_ID, output=str(zip_path), quiet=False)
    with zipfile.ZipFile(zip_path) as archive:
        archive.extractall(LOCAL_DIR)
if MASKS_ZIP.is_file() and not (PREPARED_DIR / "derived" / "inpainting_v1").is_dir():
    with zipfile.ZipFile(MASKS_ZIP) as archive:
        archive.extractall(LOCAL_DIR)


def read_csv(path):
    with Path(path).open(encoding="utf-8-sig", newline="") as stream:
        return list(csv.DictReader(stream))


def inpaint_manifest(split):
    path = PREPARED_DIR / "derived" / "inpainting_v1" / f"inpainting_{split}.csv"
    return path if path.is_file() else None


counts = {split: len(read_csv(PREPARED_DIR / "manifests" / f"{split}.csv")) for split in ("train", "val", "test")}
print("Images per split:", counts)
print("Fixed inpainting masks:", "Week 1 files" if inpaint_manifest("val") else
      "not found; regenerated with the same seeded generator")

## 4 · Install the fine-tuned weights

`scripts/install_checkpoints.py` copies each file to the place the app expects (`checkpoints/…`, `plantsr_sr/weights/`; all ignored by Git), checks the SHA-256 recorded in `configs/app.json` or in Peng's manifest, and confirms that the repository's `plantsr_sr` code is identical to the code in Peng's bundle.

In [ ]:
# Install the checkpoints and list what the app will use.
args = [sys.executable, "scripts/install_checkpoints.py"]
if DENOISE_SRC:
    args += ["--denoise", str(DENOISE_SRC)]
if COLORIZE_SRC:
    args += ["--colorize", str(COLORIZE_SRC)]
args += ["--sr", str(SR_SRC)] if SR_SRC else ["--download-sr"]
if INPAINT_LORA_SRC:
    args += ["--inpaint-lora", INPAINT_LORA_SRC]
done = subprocess.run(args, capture_output=True, text=True)
print(done.stdout[-5000:])
if done.returncode != 0:
    print(done.stderr[-3000:])
    raise RuntimeError("Installing the checkpoints failed; see the message above.")

In [ ]:
# Build the registry exactly as the app does (models load lazily, on first use).
from app.app import model_summary
from src.inference.factory import build_registry, load_config

config = load_config(CODE_DIR / "configs" / "app.json")
for key, variable in (("model_id", "PLANTSR_MODEL_ID"), ("work_res", "PLANTSR_WORK_RES"), ("steps", "PLANTSR_SD_STEPS")):
    if env(variable):  # only used for quick dry runs with a tiny model
        config["inpainting"][key] = int(env(variable)) if key != "model_id" else env(variable)
registry = build_registry(config)
print(model_summary(registry))
DEFAULTS = {t["task"]: t["default_method"] for t in registry.describe()}
expected = {"denoising": "restormer_finetuned", "colorization": "ddcolor_finetuned", "super_resolution": "hat_l"}
wrong = {task: DEFAULTS[task] for task, method in expected.items() if DEFAULTS[task] != method}
if wrong:
    display(Markdown(f"**Warning:** these tasks are not using the fine-tuned model: `{wrong}`. Check section 4."))

## 5 · Validation through the app vs. the owners' numbers

Each validation crop is degraded exactly as in its owner's notebook (seeded Gaussian noise σ = 15; Pillow bicubic ×4; CIE Lab L\*; the fixed Week 1 masks), passed through `registry.restore()`, and scored. The comparison uses the full validation split, so leave `PARITY_LIMIT = 0` for the real check. Small differences are expected from GPU precision (the training runs used bf16 on a larger GPU; a T4 runs fp16).

In [ ]:
# Score every method on the validation split through restore().
import numpy as np
import pandas as pd
from skimage.color import deltaE_ciede2000, rgb2lab
from skimage.metrics import peak_signal_noise_ratio, structural_similarity

from src.data.benchmark_examples import BenchmarkExamples
from src.evaluation.inpaint_metrics import inpaint_metrics, result_rows


def score(task, ref, out, mask=None):
    if task == "inpainting":
        m = inpaint_metrics(out, ref, mask)
        return {k: m[k] for k in ("masked_psnr", "masked_ssim", "boundary_l1", "full_psnr_supplementary")}
    if task == "super_resolution":  # Peng: float images, Gaussian SSIM
        r, o = ref.astype(np.float32) / 255, out.astype(np.float32) / 255
        return {"psnr": float(peak_signal_noise_ratio(r, o, data_range=1.0)),
                "ssim": float(structural_similarity(r, o, data_range=1.0, channel_axis=-1, gaussian_weights=True,
                                                    sigma=1.5, use_sample_covariance=False))}
    scores = {"psnr": float(peak_signal_noise_ratio(ref, out, data_range=255)),  # Jinyuan
              "ssim": float(structural_similarity(ref, out, channel_axis=2, data_range=255))}
    if task == "colorization":
        scores["delta_e00"] = float(deltaE_ciede2000(rgb2lab(ref), rgb2lab(out)).mean())
    return scores


def available(task, methods):
    names = {m["method"] for t in registry.describe() if t["task"] == task for m in t["methods"]}
    return [m for m in methods if m in names]


SD_SELECTED = f"sd15_{config['inpainting']['work_res']}_prompt"  # Week 1 name of the selected SD setting
PLAN = {
    "denoising": available("denoising", ["restormer_finetuned", "opencv_nlmeans"]),
    "colorization": available("colorization", ["ddcolor_finetuned", "grayscale_rgb"]),
    "super_resolution": available("super_resolution", ["hat_l", "real_esrgan", "hat_l_pretrained", "bicubic_x4"]),
    "inpainting": available("inpainting", ["sd15_finetuned", "sd15_pretrained", "opencv_telea"]),
}


def run_split(split, limit=0, inpaint_ids=None):
    examples = BenchmarkExamples(PREPARED_DIR, split, inpaint_manifest(split), limit=limit or None)
    rows = []
    for task, methods in PLAN.items():
        ids = [s for s in examples.ids() if inpaint_ids is None or s in inpaint_ids] if task == "inpainting" else examples.ids()
        for method in methods:
            start = time.perf_counter()
            for sid in ids:
                image, mask = examples.degraded(sid, task)
                res = registry.restore(image, task, mask, seed=SEED, method=method)
                scores = score(task, examples.reference(sid), np.asarray(res.image), mask) if res.success else None
                rows += result_rows(sid, method, scores, task=task, checkpoint_revision=res.model_version, seed=SEED,
                                    inference_seconds=res.elapsed_seconds, status="ok" if res.success else "error",
                                    error_note=res.error or "")
            print(f"{split} · {task:16s} {method:20s} {len(ids):4d} crops · {time.perf_counter() - start:6.1f}s")
    return examples, pd.DataFrame(rows)


def summarize(df):
    ok = df[df.status == "ok"].copy()
    ok["metric_value"] = ok["metric_value"].astype(float)
    table = ok.pivot_table(index=["task", "method"], columns="metric_name", values="metric_value", aggfunc="mean")
    table["crops"] = ok.groupby(["task", "method"]).sample_id.nunique()
    table["errors"] = df[df.status != "ok"].groupby(["task", "method"]).sample_id.nunique()
    return table.fillna({"errors": 0})


# Inpainting: the first N_INPAINT validation crops scored in Week 1, so the numbers can be compared.
selection = pd.read_csv(SELECTION_CSV) if SELECTION_CSV.is_file() else None
inpaint_ids = (sorted(selection[selection.method == SD_SELECTED].sample_id.unique())[:N_INPAINT]
               if selection is not None else None)
if inpaint_ids is None:
    print("Week 1 model_selection_val.csv not found; inpainting uses the first validation crops without a comparison.")
val_examples, val_rows = run_split("val", PARITY_LIMIT, set(inpaint_ids) if inpaint_ids else None)
if inpaint_ids is None:
    keep = sorted(val_rows[val_rows.task == "inpainting"].sample_id.unique())[:N_INPAINT]
    val_rows = val_rows[(val_rows.task != "inpainting") | val_rows.sample_id.isin(keep)]
val_rows.to_csv(OUTPUT_DIR / "integration_val_results.csv", index=False)
val_summary = summarize(val_rows)
val_summary.to_csv(OUTPUT_DIR / "integration_val_summary.csv")
display(val_summary.round(4))

In [ ]:
# Compare with the numbers each owner reported for the same checkpoints.
TARGETS = [  # task, method, metric, reported value, tolerance, where it was reported
    ("denoising", "restormer_finetuned", "psnr", 36.534, 0.10, "Jinyuan · val σ=15 · best checkpoint"),
    ("colorization", "ddcolor_finetuned", "delta_e00", 11.383, 0.10, "Jinyuan · val · best checkpoint"),
    ("super_resolution", "hat_l", "psnr", 33.5746, 0.02, "Peng · val · reports/super_resolution/val"),
    ("super_resolution", "real_esrgan", "psnr", 31.8204, 0.02, "Peng · val"),
    ("super_resolution", "hat_l_pretrained", "psnr", 33.5577, 0.02, "Peng · val (before fine-tuning)"),
    ("super_resolution", "bicubic_x4", "psnr", 30.3234, 0.01, "Peng · val"),
]
if selection is not None and inpaint_ids:
    same = selection[selection.sample_id.isin(inpaint_ids) & (selection.metric_name == "masked_psnr")]
    for week1_name, method, tol in ((SD_SELECTED, "sd15_pretrained", 0.5), ("opencv_telea", "opencv_telea", 1e-4)):
        values = same[same.method == week1_name].metric_value.astype(float)
        if len(values):
            TARGETS.append(("inpainting", method, "masked_psnr", float(values.mean()), tol,
                            f"Ali · Week 1 selection · same {len(values)} crops"))

parity = []
for task, method, metric, reported, tol, source in TARGETS:
    if (task, method) not in val_summary.index or metric not in val_summary.columns:
        parity.append(dict(task=task, method=method, metric=metric, app=None, reported=reported, diff=None,
                           check="not run", source=source))
        continue
    value = float(val_summary.loc[(task, method), metric])
    full = PARITY_LIMIT == 0 or task == "inpainting"
    check = ("✓ match" if abs(value - reported) <= tol else "✗ MISMATCH") if full else "subset only"
    parity.append(dict(task=task, method=method, metric=metric, app=round(value, 4), reported=reported,
                       diff=round(value - reported, 4), check=check, source=source))
parity = pd.DataFrame(parity)
parity.to_csv(OUTPUT_DIR / "parity_check.csv", index=False)
display(parity)
PARITY_OK = bool((parity.check == "✓ match").all())
display(Markdown("**All tasks match their owners' numbers.**" if PARITY_OK else
                 "**Not everything matches yet.** A mismatch means the app's pre/post-processing differs from the "
                 "training notebook; check the wrapper in `src/inference/` with the task owner before using these numbers."))

## 6 · Test split (once, with frozen models)

Run this only after every task owner confirms their final checkpoint, including the full inpainting LoRA fine-tune. It scores the test split through the same code path and writes `integration_test_results.csv` and `integration_test_summary.csv` for the report.

In [ ]:
# Final test-split evaluation (set PLANTSR_RUN_TEST=1 or RUN_TEST = True).
if RUN_TEST:
    test_examples, test_rows = run_split("test")
    test_rows.to_csv(OUTPUT_DIR / "integration_test_results.csv", index=False)
    test_summary = summarize(test_rows)
    test_summary.to_csv(OUTPUT_DIR / "integration_test_summary.csv")
    display(test_summary.round(4))
else:
    test_summary = None
    print("Skipped (RUN_TEST is False).")

## 7 · Visual check

In [ ]:
# One validation crop per task: app input, default-model output, reference.
import matplotlib.pyplot as plt

sid = val_examples.ids()[0]
fig, axes = plt.subplots(4, 3, figsize=(9, 12))
for row, task in enumerate(PLAN):
    image, mask = val_examples.degraded(inpaint_ids[0] if task == "inpainting" and inpaint_ids else sid, task)
    ref_id = inpaint_ids[0] if task == "inpainting" and inpaint_ids else sid
    res = registry.restore(image, task, mask, seed=SEED)
    for col, (title, picture) in enumerate((("input", image), (f"{res.method}", res.image),
                                            ("reference", val_examples.reference(ref_id)))):
        axes[row, col].imshow(picture)
        axes[row, col].set_title(f"{task}\n{title}" if col == 0 else title, fontsize=9)
        axes[row, col].axis("off")
plt.tight_layout()
fig.savefig(OUTPUT_DIR / "integration_examples.png", dpi=110)
plt.show()

## 8 · Automated checks

In [ ]:
# Run the repository's tests (uses the installed checkpoints; about a minute on CPU/GPU).
tests = subprocess.run([sys.executable, "-m", "pytest", "-q", "-p", "no:cacheprovider", "tests"],
                       capture_output=True, text=True)
print(tests.stdout[-4000:])
TESTS_OK = tests.returncode == 0
if not TESTS_OK:
    print(tests.stderr[-2000:])
    raise RuntimeError("Some tests failed; see the output above.")

## 9 · Start the web app

The app runs in this notebook and opens through Colab's proxy, which is fine for development and for teammates to try it while the runtime is up. For user testing and the demo, deploy it on a GPU PC as described in `docs/deployment.md`. Job and feedback logs from this session are written to `app_integration/logs/` in Drive.

In [ ]:
# Start the Flask app in a background thread and open it through Colab's proxy.
import threading

from werkzeug.serving import make_server

from app.app import create_app

if globals().get("APP_SERVER") is not None:
    APP_SERVER.shutdown()
app_examples = BenchmarkExamples(PREPARED_DIR, "val", inpaint_manifest("val"))
flask_app = create_app(registry, app_examples, log_dir=OUTPUT_DIR / "logs")
APP_SERVER = make_server("0.0.0.0" if IN_COLAB else "127.0.0.1", APP_PORT, flask_app, threaded=True)
threading.Thread(target=APP_SERVER.serve_forever, daemon=True, name="plantsr-app").start()
if IN_COLAB:
    from google.colab import output
    from google.colab.output import eval_js
    print("Open the app in a new tab:", eval_js(f"google.colab.kernel.proxyPort({APP_PORT})"))
else:
    print(f"App running at http://127.0.0.1:{APP_PORT}")

In [ ]:
# API smoke test: one benchmark job per task with the defaults, one upload with a mask, one feedback entry.
import io
import urllib.request

from PIL import Image

client = flask_app.test_client()


def wait_for(job_id, timeout=900):
    deadline = time.time() + timeout
    while time.time() < deadline:
        job = client.get(f"/api/jobs/{job_id}").get_json()
        if job["status"] in ("done", "error"):
            return job
        time.sleep(0.5)
    raise TimeoutError(job_id)


smoke = []
example_id = app_examples.ids()[0]
for task in PLAN:
    job = wait_for(client.post("/api/jobs", data={"task": task, "source": "example", "example_id": example_id,
                                                  "session": "notebook", "tester": "smoke"}).get_json()["id"])
    smoke.append({"job": f"example · {task}", "status": job["status"], "method": job["method"],
                  "output": job["result"]["output_size"], "seconds": job["result"]["elapsed_seconds"],
                  "metrics": {k: round(v, 3) for k, v in (job.get("metrics") or {}).items() if isinstance(v, float)},
                  "error": job["error"]})

photo = Image.fromarray(val_examples.reference(example_id)).resize((480, 360))
jpg = io.BytesIO(); photo.save(jpg, "JPEG", quality=95)
mask = np.zeros((360, 480), np.uint8); mask[120:200, 150:330] = 255
png = io.BytesIO(); Image.fromarray(mask).save(png, "PNG")
job = wait_for(client.post("/api/jobs", content_type="multipart/form-data", data={
    "task": "inpainting", "source": "upload", "session": "notebook", "tester": "smoke",
    "image": (io.BytesIO(jpg.getvalue()), "photo.jpg"), "mask": (io.BytesIO(png.getvalue()), "mask.png")}).get_json()["id"])
result = np.asarray(Image.open(io.BytesIO(client.get("/" + job["output_url"]).data)).convert("RGB"))
kept = bool(np.array_equal(result[mask == 0], np.asarray(Image.open(io.BytesIO(jpg.getvalue())).convert("RGB"))[mask == 0]))
smoke.append({"job": "upload · inpainting", "status": job["status"], "method": job["method"],
              "output": job["result"]["output_size"], "seconds": job["result"]["elapsed_seconds"],
              "metrics": {"visible_pixels_unchanged": kept}, "error": job["error"]})
# Peng's PR #2 check: one 64 × 64 upload per super-resolution model must return a 256 × 256 PNG.
small = io.BytesIO(); photo.resize((64, 64), Image.Resampling.BICUBIC).save(small, "PNG")
for method in [m for m in PLAN["super_resolution"] if not m.endswith("_pretrained")]:
    sr_job = wait_for(client.post("/api/jobs", content_type="multipart/form-data", data={
        "task": "super_resolution", "method": method, "source": "upload", "session": "notebook", "tester": "smoke",
        "image": (io.BytesIO(small.getvalue()), "small.png")}).get_json()["id"])
    png_ok = sr_job["status"] == "done" and Image.open(io.BytesIO(client.get("/" + sr_job["output_url"]).data)).size == (256, 256)
    smoke.append({"job": f"upload 64×64 · super_resolution", "status": sr_job["status"] if png_ok else "error",
                  "method": sr_job["method"], "output": (sr_job.get("result") or {}).get("output_size"),
                  "seconds": (sr_job.get("result") or {}).get("elapsed_seconds"), "metrics": {}, "error": sr_job["error"]})
# Pipeline: denoising -> colorization -> inpainting on the same upload, each step on the previous result.
steps = [{"task": t} for t in ("denoising", "colorization", "inpainting")]
pipe = wait_for(client.post("/api/jobs", content_type="multipart/form-data", data={
    "task": "pipeline", "steps": json.dumps(steps), "source": "upload", "session": "notebook", "tester": "smoke",
    "image": (io.BytesIO(jpg.getvalue()), "photo.jpg"), "mask": (io.BytesIO(png.getvalue()), "mask.png")}).get_json()["id"])
pipe_ok = pipe["status"] == "done" and len(pipe.get("step_urls", [])) == 3
if pipe_ok:
    before = np.asarray(Image.open(io.BytesIO(client.get("/" + pipe["step_urls"][1]).data)).convert("RGB"))
    after = np.asarray(Image.open(io.BytesIO(client.get("/" + pipe["output_url"]).data)).convert("RGB"))
    pipe_ok = bool(np.array_equal(before[mask == 0], after[mask == 0]))
smoke.append({"job": "upload · pipeline", "status": pipe["status"] if pipe_ok else "error", "method": pipe["method"],
              "output": (pipe.get("result") or {}).get("output_size"),
              "seconds": (pipe.get("result") or {}).get("elapsed_seconds"),
              "metrics": {"steps": len(pipe.get("step_urls", [])), "inpainting_only_changed_mask": pipe_ok},
              "error": pipe["error"]})
feedback = client.post("/api/feedback", json={"job_id": job["id"], "rating": 5, "comment": "notebook smoke test",
                                              "tester": "smoke", "session": "notebook"})
display(pd.DataFrame(smoke))
with urllib.request.urlopen(f"http://127.0.0.1:{APP_PORT}/api/health", timeout=10) as reply:
    HEALTH = json.loads(reply.read())
print("Live server:", {k: HEALTH[k] for k in ("device", "gpu", "loaded_models", "default_models")})
SMOKE_OK = all(r["status"] == "done" for r in smoke) and kept and feedback.status_code == 201
if not SMOKE_OK:
    raise RuntimeError("An API smoke test failed; see the table above.")

To stop the server, run `APP_SERVER.shutdown()`. It also stops when the runtime disconnects.

## 10 · Save the integration summary

In [ ]:
# Write integration_summary.json next to the CSVs in Drive.
summary = {
    "commit": COMMIT, "branch": BRANCH, "gpu": GPU, "seed": SEED,
    "models": {t["task"]: {"default": t["default_method"],
                           "methods": {m["method"]: m["kind"] for m in t["methods"]},
                           "unavailable": t["unavailable"]} for t in registry.describe()},
    "validation": {"crops": len(val_examples), "parity_limit": PARITY_LIMIT, "inpainting_crops": N_INPAINT,
                   "parity_ok": PARITY_OK, "parity": parity.to_dict("records")},
    "test_run": RUN_TEST, "tests_ok": TESTS_OK, "smoke_ok": SMOKE_OK,
}
(OUTPUT_DIR / "integration_summary.json").write_text(json.dumps(summary, indent=2, default=str), encoding="utf-8")
for path in sorted(OUTPUT_DIR.glob("*")):
    print(path.name)

## Status and next steps

- If section 5 shows a mismatch, fix the wrapper with the task owner before quoting any app numbers.
- Inpainting still uses pretrained SD 1.5 until the full LoRA fine-tune (C2) is trained and selected on validation; then pass its folder as `PLANTSR_INPAINT_LORA_SRC` and rerun sections 4–5.
- When all four models are final, run section 6 once for the report's test-split table.
- User testing (3–5 volunteers) uses the deployed app; the protocol and log files are described in `docs/deployment.md`.